# Fusion corpus_haineux ↔ corpus_emotion_haineux

## Algorithme à deux indices

Un indice `i_h` parcourt `corpus_haineux`, un indice `i_e` parcourt `corpus_emotion_haineux`. Les deux corpus sont **ordonnés de façon cohérente**, ce qui permet une traversée synchronisée.

### Trois cas selon la relation entre `tweet[i_h]` et `text[i_e]`

In [ ]:
import pandas as pd
import ast

FOLDER_BRUT = "corpus_brut_haineux"
# ── Chargement ───────────────────────────────────────────────────────────────
PATH_HAINEUX = 'corpus_haineux.csv'          # séparateur ;
PATH_EMOTION = 'corpus_emotion_haineux.csv'  # séparateur ,

df_h = pd.read_csv(f"{FOLDER_BRUT}/{PATH_HAINEUX}", sep=';')
df_e = pd.read_csv(PATH_EMOTION, sep=',')

# Normalisation : strip + espaces multiples/sauts de ligne → espace simple
df_h['tweet_clean'] = df_h['tweet'].str.strip().str.replace(r'\s+', ' ', regex=True)
df_e['text_clean']  = df_e['text'].str.strip().str.replace(r'\s+', ' ', regex=True)

print(f'corpus_haineux : {len(df_h)} tweets')
print(f'corpus_emotion : {len(df_e)} phrases')

In [ ]:
# ── Parsing des colonnes liste ────────────────────────────────────────────────

def safe_parse_list(val):
    """Convertit \"['a','b']\" en liste Python. Retourne [] si vide/NaN."""
    if pd.isna(val) or str(val).strip() in ('', '[]'):
        return []
    try:
        result = ast.literal_eval(str(val))
        return result if isinstance(result, list) else []
    except Exception:
        return []

for col in ['Tokens', 'Lemmas', 'Cats']:
    df_e[col] = df_e[col].apply(safe_parse_list)

n_tok = df_e['Tokens'].apply(len).sum()
print(f'Total tokens à distribuer : {n_tok}')
df_e[df_e['Tokens'].apply(len) > 0][['file_id', 'text_clean', 'Tokens', 'Cats']].head(5)

In [ ]:
# ── Fonctions utilitaires ─────────────────────────────────────────────────────

def no_relation(te: str, tw: str) -> bool:

    return te != tw and te not in tw and tw not in te


def find_next_matching_tweet(df_h, df_e, i_h_start: int, i_e: int,
                              max_lookahead: int = 10):

    te = df_e.iloc[i_e]['text_clean']
    for k in range(max_lookahead):
        idx = i_h_start + k
        if idx >= len(df_h):
            break
        tw = df_h.iloc[idx]['tweet_clean']
        if not no_relation(te, tw):
            return idx
    return None




In [ ]:
# ── Algorithme principal à deux indices ───

i_h, i_e = 0, 0
n_h, n_e = len(df_h), len(df_e)

# Accumulateurs de features par tweet
assigned = {
    i: {'Tokens': [], 'Lemmas': [], 'Cats': [], 'e_ids': [], 'cas': None}
    for i in range(n_h)
}

while i_h < n_h and i_e < n_e:
    tw   = df_h.iloc[i_h]['tweet_clean']
    te   = df_e.iloc[i_e]['text_clean']
    toks = df_e.iloc[i_e]['Tokens']
    lems = df_e.iloc[i_e]['Lemmas']
    cats = df_e.iloc[i_e]['Cats']
    eid  = df_e.iloc[i_e]['file_id']

    # match exact
    if te == tw:
        assigned[i_h]['cas'] = 'exact'
        assigned[i_h]['Tokens'] += toks
        assigned[i_h]['Lemmas'] += lems
        assigned[i_h]['Cats']   += cats
        assigned[i_h]['e_ids'].append(eid)
        i_h += 1
        i_e += 1

    #émotion INCLUS tweet tweet découpé en
    elif te in tw:
        assigned[i_h]['cas'] = 'e_in_tw'
        assigned[i_h]['Tokens'] += toks
        assigned[i_h]['Lemmas'] += lems
        assigned[i_h]['Cats']   += cats
        assigned[i_h]['e_ids'].append(eid)
        i_e += 1
        # Avancer tweet seulement si la prochaine émotion ne lui appartient plus
        if i_e < n_e and no_relation(df_e.iloc[i_e]['text_clean'], tw):
            i_h += 1

    # tweet inclus émotion (plusieurs tweets regroupés dans une phrase)
    elif tw in te:
        assigned[i_h]['cas'] = 'tw_in_e'
        # Ne garder que les features dont le token est présent dans ce tweet
        for j, tok in enumerate(toks):
            if tok.lower() in tw.lower():
                assigned[i_h]['Tokens'].append(tok)
                assigned[i_h]['Lemmas'].append(lems[j] if j < len(lems) else None)
                assigned[i_h]['Cats'].append(cats[j]   if j < len(cats) else None)
        assigned[i_h]['e_ids'].append(eid)
        i_h += 1  # avancer tweet — PAS émotion (elle couvre encore les tweets suivants)

    # ── ORPHAN : aucune relation entre tweet courant et phrase courante ─
    else:
        # Chercher si un tweet proche (fantôme) correspond à cette émotion
        next_h = find_next_matching_tweet(df_h, df_e, i_h, i_e, max_lookahead=10)
        if next_h is not None and next_h != i_h:
            # Sauter les tweets intermédiaires (no_match : absents de corpus_emotion)
            i_h = next_h
        else:
            # Phrase orpheline (chevauchement de frontière) → avancer émotion
            i_e += 1

print(f'Traitement terminé : i_h={i_h}/{n_h}, i_e={i_e}/{n_e}')

In [ ]:
# ── Construction du dataframe final ──────────────────────────────────────────

rows = []
for i in range(n_h):
    feat = assigned[i]
    rows.append({
        'tweet'       : df_h.iloc[i]['tweet'],
        'label'       : df_h.iloc[i]['label'],
        'Tokens'      : feat['Tokens'],
        'Lemmas'      : feat['Lemmas'],
        'Cats'        : feat['Cats'],
        'emotion_ids' : feat['e_ids'],
        'cas'         : feat['cas'] or 'no_match',
    })

df_merged = pd.DataFrame(rows)

print('=== Distribution des cas ===')
print(df_merged['cas'].value_counts())
print()

avec = df_merged['Tokens'].apply(len).gt(0).sum()
total_tok = df_merged['Tokens'].apply(len).sum()
total_in_e = df_e['Tokens'].apply(len).sum()

print(f'Tweets avec au moins 1 feature : {avec} / {n_h}')
print(f'Tokens distribués               : {total_tok} / {total_in_e}')
print(f'Tokens non distribués (orphans) : {total_in_e - total_tok}')

df_merged.head(10)

In [ ]:
df_merged.head(200).tail(10)

In [ ]:
# ── Export ────────────────────────────────────────────────────────────────────

df_export = df_merged.drop(columns=['cas']).copy()  # retirer la colonne debug
for col in ['Tokens', 'Lemmas', 'Cats', 'emotion_ids']:
    df_export[col] = df_export[col].apply(str)

df_export.to_csv('corpus_haine_emotion.csv', sep=',', index=False, encoding='utf-8-sig')
print("Fichier 'corpus_haine_emotion.csv' exporté (sep=',').")
df_export.head(5)